# Wheat & Sunflower MIC Recomputation — Correct Crop Calendars

**Standalone notebook** — recomputes MIC permutation p-values for ONLY two variables (Wheat and Sunflower) using their correct crop calendar matching:

- **Wheat** = Nov(Y−1) + Dec(Y−1) + Jan(Y) + Feb(Y) + Mar(Y) + Apr(Y) + May(Y) + Jun(Y)  *(8 months crop year)*
- **Sunflower** = Apr(Y) + May(Y) + Jun(Y) + Jul(Y) + Aug(Y)  *(5 months vegetative season; September excluded since the crop is harvested before September precipitation accumulates)*

The previous notebook used Oct–May for Wheat (off by one month) and Apr–Sep for Sunflower (one extra month). Verification on the precipitation CSVs confirmed the correct conventions: Wheat column = Nov–Jun crop sum (59/59 exact match across all stations), Sunflower column = Apr–Aug sum (60/60 exact match across all stations).

**This corrects the index-matching for two variables only.** All other variables (12 monthly + 4 seasonal + Annual = 17 variables) remain untouched — those used calendar-year matching which is correct.

**Combinations to recompute:** 8 stations × 2 variables × 4 indices = **64 combinations**.

**Estimated runtime:** ~5 minutes on Colab CPU.

**Author:** Cantekin Kıvrak (TAGEM Kırklareli)  
**Generated:** 2026-04-30


## 1. Setup

In [ ]:
!pip install -q tqdm openpyxl 2>&1 | tail -1

import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

RNG_SEED = 20260430
np.random.seed(RNG_SEED)
print('Setup complete.')


## 2. Pure-Python MINE class

Equipartition implementation matching the one used in the original notebook (since `minepy` failed to build on Colab Python 3.12). MIC values are slightly lower than minepy (~5–15%) but **permutation p-values are valid** because the same statistic is used for both observed and permuted MIC.


In [ ]:
class MINE:
    """Approximate MIC using equipartition binning."""
    def __init__(self, alpha=0.6, c=15, est='mic_approx'):
        self.alpha = alpha
        self.c = c
        self._mic = 0.0
    
    def compute_score(self, x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        n = len(x)
        if n < 4:
            self._mic = 0.0
            return
        B = max(int(n ** self.alpha), 4)
        best = 0.0
        for a in range(2, B // 2 + 1):
            max_b = min(B // a, B - 1)
            for b in range(2, max_b + 1):
                if a * b > B:
                    break
                x_bins = self._equipartition(x, a)
                y_bins = self._equipartition(y, b)
                mi = self._mutual_info(x_bins, y_bins, a, b)
                log_min = np.log(min(a, b))
                if log_min > 0:
                    normed = mi / log_min
                    if normed > best:
                        best = normed
        self._mic = float(min(best, 1.0))
    
    def mic(self):
        return self._mic
    
    @staticmethod
    def _equipartition(x, n_bins):
        ranks = stats.rankdata(x, method='ordinal') - 1
        return np.minimum((ranks * n_bins // len(x)).astype(int), n_bins - 1)
    
    @staticmethod
    def _mutual_info(x_bins, y_bins, a, b):
        n = len(x_bins)
        flat = x_bins * b + y_bins
        counts = np.bincount(flat, minlength=a * b).reshape(a, b).astype(float)
        p = counts / n
        px = p.sum(axis=1, keepdims=True)
        py = p.sum(axis=0, keepdims=True)
        pxpy = px * py
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where((p > 0) & (pxpy > 0), p / pxpy, 1.0)
            log_term = np.log(ratio)
            log_term = np.where(np.isfinite(log_term), log_term, 0.0)
        mi = float((p * log_term).sum())
        return max(0.0, mi)


def compute_mic(x, y, alpha=0.6, c=15):
    m = MINE(alpha=alpha, c=c, est='mic_approx')
    m.compute_score(x, y)
    return m.mic()

# Sanity test
print('MIC for random noise (should be ~0.1-0.4):', round(compute_mic(np.random.randn(60), np.random.randn(60)), 4))
print('MIC for perfect linear (should be ~1.0):', round(compute_mic(np.linspace(0,1,60), np.linspace(0,1,60)), 4))


## 3. Upload data

Upload all 12 files (8 precip CSVs + 4 teleconnection xlsx).

In [ ]:
from google.colab import files
print('Select all 12 data files (Ctrl/Cmd+click for multi-select):')
uploaded = files.upload()
print(f'\nUploaded {len(uploaded)} files.')


## 4. Load precipitation data

In [ ]:
STATION_FILES = {
    'Edirne':     'edirne_prcp_filled.csv',
    'Kırklareli': 'kirklareli_prcp_filled.csv',
    'Lüleburgaz': 'luleburgaz_prcp_filled.csv',
    'Uzunköprü':  'uzunkopru_prcp_filled.csv',
    'İpsala':     'ipsala_prcp_filled.csv',
    'Çorlu':      'corlu_prcp_filled.csv',
    'Tekirdağ':   'tekirdagp_filled.csv',
    'Sarıyer':    'sariyer_prcp_filled.csv',
}

PRECIP_YEARS = list(range(1965, 2025))
TARGET_VARS = ['Wheat', 'Sunflower']  # Only these two variables

precip_data = {}
for station, fname in STATION_FILES.items():
    df = pd.read_csv(fname).set_index('year')
    df = df.loc[df.index.isin(PRECIP_YEARS)]
    for var in TARGET_VARS:
        precip_data[f'{station}_{var}'] = df[var].values

precip_df = pd.DataFrame(precip_data, index=PRECIP_YEARS)
print(f'Loaded {precip_df.shape[1]} series ({len(STATION_FILES)} stations × {len(TARGET_VARS)} variables) × {precip_df.shape[0]} years')


## 5. Load teleconnection indices (with 1964 data)

In [ ]:
INDEX_YEARS = list(range(1964, 2025))   # 1964 needed for crop-year matching of Wheat 1965

def load_monthly_index(path, year_col='Year'):
    df = pd.read_excel(path)
    df = df.drop_duplicates(subset=[year_col], keep='first')
    df = df.set_index(year_col)
    df = df.loc[df.index.isin(INDEX_YEARS)]
    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    return df[months]

def load_enso_index(path, year_col='Year'):
    df = pd.read_excel(path)
    df = df.drop_duplicates(subset=[year_col], keep='first')
    df = df.set_index(year_col)
    df = df.loc[df.index.isin(INDEX_YEARS)]
    mapping = {'DJF':'Jan','JFM':'Feb','FMA':'Mar','MAM':'Apr',
               'AMJ':'May','MJJ':'Jun','JJA':'Jul','JAS':'Aug',
               'ASO':'Sep','SON':'Oct','OND':'Nov','NDJ':'Dec'}
    out = pd.DataFrame(index=df.index)
    for window, month in mapping.items():
        out[month] = df[window].values
    return out

# Handle ENSO filename variants from upload
import os
enso_path = None
for fname in os.listdir('.'):
    fname_lower = fname.lower()
    if fname.endswith('.xlsx') and ('nino' in fname_lower or 'enso' in fname_lower):
        enso_path = fname
        break
print(f'ENSO file: {enso_path}')

tele_indices = {
    'AO':   load_monthly_index('AO.xlsx'),
    'NAO':  load_monthly_index('NAO.xlsx'),
    'PNA':  load_monthly_index('PNA.xlsx'),
    'ENSO': load_enso_index(enso_path),
}
for name, df in tele_indices.items():
    print(f'  {name:5s}: {df.shape[0]} years × 12 months ({df.index.min()}–{df.index.max()})')


## 6. CORRECTED variable-month matching

The key correction: Wheat is Nov(Y−1) to Jun(Y) — 8 months of crop year — and Sunflower is Apr to Aug (5 months, September excluded). The precipitation CSV's Wheat and Sunflower columns use these conventions, verified by exact arithmetic match across all 60 years × 8 stations.


In [ ]:
# Variable → list of (month, year_offset) — CORRECTED
VARIABLE_TO_MONTHS = {
    # Wheat: Nov(Y-1) through Jun(Y) — 8 months
    'Wheat': [('Nov', -1), ('Dec', -1),
              ('Jan', 0), ('Feb', 0), ('Mar', 0), ('Apr', 0), ('May', 0), ('Jun', 0)],
    # Sunflower: Apr(Y) through Aug(Y) — 5 months (Sep harvested-excluded)
    'Sunflower': [('Apr', 0), ('May', 0), ('Jun', 0), ('Jul', 0), ('Aug', 0)],
}


def index_series_for_variable(index_name, variable):
    df_idx = tele_indices[index_name]
    pattern = VARIABLE_TO_MONTHS[variable]
    out = np.zeros(len(PRECIP_YEARS))
    for i, year in enumerate(PRECIP_YEARS):
        values = []
        for month, offset in pattern:
            target_year = year + offset
            if target_year in df_idx.index:
                values.append(df_idx.loc[target_year, month])
            else:
                values.append(np.nan)
        out[i] = np.nanmean(values) if any(not np.isnan(v) for v in values) else np.nan
    return out


# Sanity check
print('Sanity check (first 3 years):')
print(f'  NAO_Wheat (Nov(Y-1)-Jun(Y)):  {np.round(index_series_for_variable("NAO", "Wheat")[:3], 3)}')
print(f'  NAO_Sunflower (Apr-Aug):       {np.round(index_series_for_variable("NAO", "Sunflower")[:3], 3)}')

# Verify Wheat 1965 uses 1964 Nov/Dec
nao_nov_1964 = tele_indices['NAO'].loc[1964, 'Nov']
nao_dec_1964 = tele_indices['NAO'].loc[1964, 'Dec']
nao_1965 = tele_indices['NAO'].loc[1965]
expected = np.mean([nao_nov_1964, nao_dec_1964,
                     nao_1965['Jan'], nao_1965['Feb'], nao_1965['Mar'],
                     nao_1965['Apr'], nao_1965['May'], nao_1965['Jun']])
actual = index_series_for_variable('NAO', 'Wheat')[0]
print(f'\nNAO Wheat 1965 manual:   {expected:.4f}')
print(f'NAO Wheat 1965 function: {actual:.4f}')
print(f'Match: {abs(expected-actual)<0.001}')


## 7. Run MIC permutation

In [ ]:
def mic_permutation(precip, index, n_perm=1000, seed=RNG_SEED):
    precip = np.asarray(precip, dtype=float)
    index = np.asarray(index, dtype=float)
    mask = ~(np.isnan(precip) | np.isnan(index))
    p, ix = precip[mask], index[mask]
    if len(p) < 10:
        return None
    
    mic_obs = compute_mic(p, ix)
    rng = np.random.default_rng(seed)
    perm = np.zeros(n_perm)
    for i in range(n_perm):
        perm[i] = compute_mic(rng.permutation(p), ix)
    
    p_value = (1.0 + np.sum(perm >= mic_obs)) / (1.0 + n_perm)
    r, _ = stats.pearsonr(p, ix)
    nl = mic_obs - r**2
    return {'mic': float(mic_obs), 'p_perm': float(p_value),
            'pearson_r': float(r), 'NL': float(nl)}


N_PERM = 1000
INDEX_NAMES = ['AO', 'NAO', 'PNA', 'ENSO']
results = []

combos = [(col, idx) for col in precip_df.columns for idx in INDEX_NAMES]
print(f'Total combinations: {len(combos)} (8 stations × 2 variables × 4 indices)')

for i, (col, idx_name) in enumerate(tqdm(combos, desc='MIC permutation')):
    station, variable = col.split('_', 1)
    precip_series = precip_df[col].values
    index_series = index_series_for_variable(idx_name, variable)
    
    res = mic_permutation(precip_series, index_series, n_perm=N_PERM, seed=RNG_SEED + i)
    if res is None:
        continue
    results.append({
        'station': station, 'variable': variable, 'index': idx_name,
        'mic': res['mic'], 'p_value_perm': res['p_perm'],
        'pearson_r': res['pearson_r'], 'NL_statistic': res['NL'],
        'sig_p010': res['p_perm'] < 0.10,
        'sig_p005': res['p_perm'] < 0.05,
        'sig_p001': res['p_perm'] < 0.01,
    })

new_mic_df = pd.DataFrame(results)
new_mic_df.to_csv('mic_wheat_sunflower_corrected.csv', index=False)
print(f'\nResults saved: {len(new_mic_df)} rows')
new_mic_df.head(10)


## 8. Comparison summary — old vs new

In [ ]:
# Compare significant counts under new vs old matching
# (Old results would need to be loaded if you want side-by-side; here we just summarize new)

print('=' * 70)
print('NEW MIC RESULTS — Wheat (Nov-Jun) and Sunflower (Apr-Aug)')
print('=' * 70)

for var in ['Wheat', 'Sunflower']:
    print(f'\n{var}:')
    sub_var = new_mic_df[new_mic_df['variable']==var]
    for idx in INDEX_NAMES:
        sub = sub_var[sub_var['index']==idx]
        n_p10 = int(sub['sig_p010'].sum())
        n_p05 = int(sub['sig_p005'].sum())
        mean_mic = sub['mic'].mean()
        nl_only = int(((sub['p_value_perm'] < 0.05) & (np.abs(sub['pearson_r']) < 0.10)).sum())
        pe_only = int(((np.abs(sub['pearson_r']) > 0.30) & (sub['p_value_perm'] >= 0.05)).sum())
        print(f'  {idx}: n_sig p≤0.10={n_p10}/8, p≤0.05={n_p05}/8, mean MIC={mean_mic:.3f}, NL-only={nl_only}, Pearson-only={pe_only}')

# Top associations
print('\n\nTop 10 strongest MIC associations (Wheat + Sunflower):')
top = new_mic_df.sort_values('p_value_perm').head(10).copy()
top['mic'] = top['mic'].round(3)
top['p_value_perm'] = top['p_value_perm'].round(4)
top['pearson_r'] = top['pearson_r'].round(3)
top['NL_statistic'] = top['NL_statistic'].round(3)
print(top[['station','variable','index','mic','p_value_perm','pearson_r','NL_statistic']].to_string(index=False))


## 9. Download

In [ ]:
from google.colab import files
files.download('mic_wheat_sunflower_corrected.csv')


## 10. Send Claude

Send the printed Section 8 summary (significant counts per index for both variables, and Top 10 associations) plus the downloaded CSV. Claude will:

1. Compute the change in Table 6 totals (most likely small shifts — 1-2 cells per index column)
2. Update Table 6 with tracked changes if needed
3. Update the §3.6 narrative if any "Wheat" or "Sunflower" specific finding changes
4. Update Figure 8 caption count if "candidate nonlinear" total changes from 11 to 10/12
5. Cross-check that no other manuscript text refers to Wheat-Sunflower MIC values that need updating
